# Соревновательные приёмы

StratifiedKFold, ансамбли, pseudo-labeling, label smoothing, gradient accumulation, AMP, early stopping, class weights.

## Импорты

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.cuda.amp import autocast, GradScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

## StratifiedKFold (sklearn-модели)

In [ ]:
N_FOLDS = 5
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(X_test))

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f'--- Fold {fold+1}/{N_FOLDS} ---')
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = ...  # твоя модель (CatBoost, XGBoost, sklearn, etc.)
    model.fit(X_tr, y_tr)  # добавить eval_set при необходимости

    oof_preds[val_idx] = model.predict(X_val)
    test_preds += model.predict(X_test) / N_FOLDS

print(f'\nOOF F1: {f1_score(y, oof_preds, average="macro"):.4f}')

## StratifiedKFold (PyTorch-модели) -- с вероятностями

In [ ]:
N_FOLDS = 5
NUM_CLASSES = 3  # <-- подставить

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
oof_probs = np.zeros((len(train_texts), NUM_CLASSES))
test_probs_folds = []

for fold, (train_idx, val_idx) in enumerate(skf.split(train_texts, train_labels)):
    print(f'\n--- Fold {fold+1}/{N_FOLDS} ---')

    # ... создать DataLoader'ы для fold ...
    # ... создать модель, optimizer, scheduler ...
    # ... train loop (сохранить лучшую модель по val_f1) ...

    # После обучения: OOF-предсказания
    # oof_probs[val_idx] = predict_proba(model, val_loader, device)
    # test_probs_folds.append(predict_proba(model, test_loader, device))
    pass

# Усреднение тестовых предсказаний по фолдам
# test_probs = np.mean(test_probs_folds, axis=0)
# test_preds = test_probs.argmax(axis=1)

# oof_preds = oof_probs.argmax(axis=1)
# print(f'OOF F1: {f1_score(train_labels, oof_preds, average="macro"):.4f}')

## Ансамбль моделей (взвешенное усреднение)

In [ ]:
# probs_model1, probs_model2, probs_model3 -- np.array [n_samples, n_classes]
# f1_model1, f1_model2, f1_model3 -- OOF F1 каждой модели

model_probs = [probs_model1, probs_model2, probs_model3]
model_f1s = np.array([f1_model1, f1_model2, f1_model3])

# Веса пропорциональны качеству
weights = model_f1s / model_f1s.sum()
print(f'Ensemble weights: {weights}')

ensemble_probs = sum(w * p for w, p in zip(weights, model_probs))
ensemble_preds = ensemble_probs.argmax(axis=1)

# Проверка на OOF
# oof_ensemble = sum(w * oof for w, oof in zip(weights, oof_probs_list))
# print(f'Ensemble OOF F1: {f1_score(y, oof_ensemble.argmax(axis=1), average="macro"):.4f}')

## Pseudo-Labeling

In [ ]:
# test_probs -- np.array [n_test, n_classes] от обученной модели
THRESHOLD = 0.90

max_probs = test_probs.max(axis=1)
confident_mask = max_probs >= THRESHOLD
pseudo_labels = test_probs.argmax(axis=1)[confident_mask]

print(f'Pseudo-labeled: {confident_mask.sum()}/{len(test_probs)} samples')

# Добавить к обучающей выборке
# if isinstance(train_texts, list):
#     pseudo_texts = [test_texts[i] for i in np.where(confident_mask)[0]]
#     augmented_texts = train_texts + pseudo_texts
#     augmented_labels = np.concatenate([train_labels, pseudo_labels])

# Для табличных данных:
# pseudo_X = X_test[confident_mask]
# augmented_X = np.vstack([X_train, pseudo_X])
# augmented_y = np.concatenate([y_train, pseudo_labels])

## Label Smoothing

In [ ]:
# Встроено в PyTorch CrossEntropyLoss
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Вместо one-hot [0, 0, 1, 0] будет [0.025, 0.025, 0.925, 0.025] при eps=0.1, K=4

## Class Weights (для несбалансированных данных)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Автоматический расчёт весов
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print(f'Class weights: {class_weights}')

criterion = nn.CrossEntropyLoss(weight=class_weights)

# Или вручную:
# counts = np.bincount(y_train)
# weights = counts.sum() / (len(counts) * counts)
# criterion = nn.CrossEntropyLoss(weight=torch.tensor(weights, dtype=torch.float32).to(device))

# Следующие трюки вам с вероятностью 99% не понадобятся на закле
но быть может в будущем вы будете решать что-нибудь на кегле

## Gradient Accumulation

In [ ]:
GRAD_ACCUM_STEPS = 4  # эффективный batch_size = batch_size * 4
scaler = GradScaler()

model.train()
optimizer.zero_grad()

for step, (batch_x, batch_y) in enumerate(train_loader):
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)

    with autocast():
        logits = model(batch_x)
        loss = criterion(logits, batch_y) / GRAD_ACCUM_STEPS

    scaler.scale(loss).backward()

    if (step + 1) % GRAD_ACCUM_STEPS == 0:
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad()
        # scheduler.step()  # если per-step scheduler

## Mixed Precision (AMP)

In [ ]:
scaler = GradScaler()

for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)
    optimizer.zero_grad()

    with autocast():
        logits = model(batch_x)
        loss = criterion(logits, batch_y)

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()

## Early Stopping (класс)
это можно просто if-ком добавить в цикл, но так симпатичнее

In [ ]:
class EarlyStopping:
    def __init__(self, patience=5, mode='max'):
        self.patience = patience
        self.counter = 0
        self.best_score = None
        self.mode = mode

    def __call__(self, score):
        if self.best_score is None:
            self.best_score = score
            return False

        improved = score > self.best_score if self.mode == 'max' else score < self.best_score
        if improved:
            self.best_score = score
            self.counter = 0
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience

# Использование:
# es = EarlyStopping(patience=5, mode='max')
# for epoch in range(num_epochs):
#     ...
#     if es(val_f1):
#         print(f'Early stopping at epoch {epoch+1}')
#         break

## Submission -- шаблон

In [ ]:
submission = pd.DataFrame({
    'id': test_ids,       # <-- поменять
    'target': test_preds  # <-- поменять
})
submission.to_csv('submission.csv', index=False)
print(f'Submission shape: {submission.shape}')
submission.head()